# Stage One: Data Collection

## Is the collector car market getting younger?

This project compares the most valuable cars sold at the key collector car auctions of the last 12 months with the same sales three years earlier. The aim is to test a claim often made about the post-pandemic market: that older cars are losing ground while cars from the 1980s and 1990s are taking over the top of the market.

**Sales compared (each against its own edition three years earlier):**

| Sale | Last 12 months | Three years earlier | Lots used |
|---|---|---|---|
| Monterey Car Week (all five houses) | August 2026 | August 2023 | Top 30 by price |
| Amelia Island (all houses) | March 2026 | March 2023 | Top 10 by price |
| RM Sotheby's Arizona | January 2026 | January 2023 | Every car over about $1m |
| RM Sotheby's Paris | January 2026 | February 2023 | Top 15 by price |

Week totals are also collected for Arizona, Paris, Amelia, Monterey and RM Sotheby's London.

## Stage One - Part One: Sources

All data comes from public results: auction-house press releases, Hagerty's live auction reports and published results lists (classic-car-auctions.info, classic.com). Prices include buyer's premium. Euro prices were converted at the rate given by the source.

Two kinds of record were collected by hand into CSV files:

- `top_lots.csv`: one row per car, with the sale, the edition, the auction house, the model year, the car and the price in US dollars.
- `sale_totals.csv`: one row per sale and edition, with the total and, where published, the sell-through rate.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
THEN, NOW = "#7FA3BF", "#B8945A"

In [2]:
lots = pd.read_csv("../data/raw/top_lots.csv")
totals = pd.read_csv("../data/raw/sale_totals.csv")

print(f"Top lots: {lots.shape[0]} rows, {lots.shape[1]} columns")
print(f"Sale totals: {totals.shape[0]} rows, {totals.shape[1]} columns")
lots.head(10)

Top lots: 145 rows, 6 columns
Sale totals: 10 rows, 6 columns


,week,period,house,year,car,price_usd
0,Monterey,2023,RM Sotheby's,1957,Jaguar XKSS,13205000
1,Monterey,2023,RM Sotheby's,1959,Ferrari 410 Superamerica Series III,6605000
2,Monterey,2023,RM Sotheby's,1937,Bugatti Type 57SC Tourer by Corsica,5395000
3,Monterey,2023,RM Sotheby's,1967,Ferrari 275 GTB/4 (McQueen),5395000
4,Monterey,2023,RM Sotheby's,2003,Ferrari Enzo,4075000
5,Monterey,2023,RM Sotheby's,1967,Ferrari 275 GTB/4 Del Rio,3800000
6,Monterey,2023,RM Sotheby's,1933,Packard Twelve Convertible Victoria,3305000
7,Monterey,2023,RM Sotheby's,1990,Ferrari F40,3305000
8,Monterey,2023,RM Sotheby's,1965,Ferrari 275 GTB/6C Alloy,3305000
9,Monterey,2023,RM Sotheby's,1935,Duesenberg Model SJ LaGrande Phaeton,3030000


In [3]:
lots.info()

<class 'pandas.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   week       145 non-null    str  
 1   period     145 non-null    int64
 2   house      145 non-null    str  
 3   year       145 non-null    int64
 4   car        145 non-null    str  
 5   price_usd  145 non-null    int64
dtypes: int64(3), str(3)
memory usage: 6.9 KB


## Stage One - Part Two: Checking the data

In [4]:
checks = {
    "missing values": int(lots.isna().sum().sum()),
    "duplicate rows": int(lots.duplicated().sum()),
    "earliest model year": int(lots["year"].min()),
    "latest model year": int(lots["year"].max()),
    "cheapest lot (USD)": f"{lots['price_usd'].min():,.0f}",
    "dearest lot (USD)": f"{lots['price_usd'].max():,.0f}",
}
pd.Series(checks, name="value").to_frame()

,value
missing values,0
duplicate rows,0
earliest model year,1912
latest model year,2025
cheapest lot (USD),"926,434"
dearest lot (USD),"42,905,000"


In [5]:
counts = lots.pivot_table(index="week", columns="period", values="car", aggfunc="count")
counts["total"] = counts.sum(axis=1)
counts

period,2023,2026,total
week,,,
Amelia,10,10,20
Arizona (RM Sotheby's),12,12,24
Monterey,40,31,71
Paris (RM Sotheby's),15,15,30


The Monterey file holds more than 30 cars for 2023, because the million-dollar lists of each house were collected in full. Stage Two keeps the 30 most valuable in each year, so that both editions are compared on equal terms.

The 2026 Monterey list leaves out the $40m Ferrari Luce charity lot, a brand-new car sold for charity, which would distort any comparison. In 2023 one RM Sotheby's car sold at an undisclosed price and is also left out.

In [6]:
totals

,week,period,total_usd_m,sell_through,basis,note
0,Arizona,2023,265.5,NaN,House totals: Barrett-Jackson 190.6 + RM 44.9 ...,Barrett-Jackson is no reserve
1,Arizona,2026,269.2,NaN,House totals: Barrett-Jackson 195.2 + RM 63.0 ...,RM up 40 percent; Bonhams down
2,Paris,2023,127.1,77.0,Three houses (RM; Bonhams; Artcurial),classic.com tally
3,Paris,2026,190.0,NaN,Four houses incl. new Gooding Christie's: EUR ...,Artcurial 57 percent sold
4,Amelia,2023,178.4,87.0,Hagerty: Gooding; RM; Broad Arrow; Bonhams,NaN
5,Amelia,2026,255.9,92.0,Hagerty March classic auctions: Broad Arrow; G...,RM moved to Miami
6,Monterey,2023,400.1,68.0,Hagerty: five houses,NaN
7,Monterey,2026,755.6,76.0,Hagerty: five houses incl. post-sale; includes...,718.3 without the Luce
8,London (RM),2022,31.6,83.0,RM Sotheby's London: GBP 27.26m,NaN
9,London (RM),2025,31.5,79.0,RM Sotheby's London: GBP 23.70m,Down 13 percent in pounds
